# 量化对照：fp16 vs AWQ vs GPTQ-Int4

对应 JD 那条「**开展模型量化实验（INT4 QAT / GPTQ / AWQ），验证精度与性能平衡**」。

做的是**训练后量化（PTQ）**——GPTQ 与 AWQ 都不需要训练，直接加载社区已量化的权重。
**QAT 不做**：那个要真训练，免费 GPU 额度撑不住。面试如实说「PTQ 做了两种，QAT 只懂原理」。

## 一个可以先写下来的预测

T4 是 **sm_75**。vLLM 的 GPTQ 快速路径 **Marlin kernel 要 sm_80 以上**，
在 T4 上会回落到老的 GPTQ kernel。所以有理由预测：

> **在 T4 上，GPTQ-Int4 的吞吐可能不如 fp16**——显存省了，速度不一定赢。

这和本仓之前那个「无 Tensor Core 卡上 fp32 反而比 fp16 快」是同一类现象：
**量化省显存 ≠ 一定更快，要看目标架构上有没有对应的高效 kernel。**

先写下预测，再跑数据验证或推翻——两种结果都照实记。

## 三个维度

| 维度 | 指标 |
|---|---|
| 速度 | TTFT p50/p99、TPOT p50、吞吐 tok/s（并发扫描） |
| 显存 | 权重占用、KV cache 可用 token 数 |
| 精度 | wikitext 困惑度（PPL）+ 与 fp16 基线的贪心输出一致率 |


## 0. 环境

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
cc = out.strip().splitlines()[-1].split(",")[-1].strip()
print("compute capability:", cc)
print()
print("sm_80 以上才有 Marlin 快速 GPTQ kernel。当前:",
      "有 Marlin" if cc.replace(".", "") >= "80" else "无 Marlin —— GPTQ 会走慢路径（正是要验证的）")


## 1. 装 vLLM

三个包一条命令一起解析（**不含 torchaudio**）。判断条件是「**全部**在场才跳过」——
之前两次栽在这个 guard 上：一次是「已装 vLLM 就跳过卸载」，一次是「缺包就跳过安装」。

2026-09-05 实测补充：`torchaudio` 与 `torch` 版本错配时，transformers 的 `audio_utils`
会 import 它并炸掉整条导入链。vLLM 不需要它，装完每次都卸。


In [ ]:
import importlib.metadata as md_, subprocess, sys

CHECK = ["vllm", "aiohttp", "torchvision"]      # 不含 torchaudio：与 torch 版本错配会炸 transformers 导入链

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")

if missing:
    print("装 vllm + aiohttp + torchvision（一条命令，约 5-10 分钟）...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "vllm", "aiohttp", "torchvision"],
                       capture_output=True, text=True)
    print("退出码:", r.returncode)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
else:
    print("三个包都在，跳过安装。")

# 2026-09-05 实测：torchaudio 与 torch 版本错配时，transformers 的 audio_utils
# 会 import 它并炸掉整条导入链。vLLM 不需要它，每次都卸。
print()
u = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
                   capture_output=True, text=True)
print("卸 torchaudio 退出码:", u.returncode)

print()
for p in CHECK + ["transformers", "torch"]:
    print("  %-16s %s" % (p, ver(p) or "（未安装）"))
print("  %-16s %s" % ("torchaudio", ver("torchaudio") or "已卸载 OK"))


## 2. 确认三个模型都存在

用 1.5B 而不是 0.5B：模型越小，量化的相对损失越难测出来，1.5B 的信号更清楚，
而且 T4 的 16 GiB 放得下 fp16 版本。

这一格只查 HuggingFace 上文件在不在，**不下载**。若某个仓库不存在或改名了，
这里会直接报出来，不会等到起服务时才失败。


In [ ]:
# huggingface_hub 不加载 torch，可以直接在主进程里查，不用起子进程
from huggingface_hub import model_info

MODELS = {
    "fp16":       "Qwen/Qwen2.5-1.5B-Instruct",
    "AWQ":        "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "GPTQ-Int4":  "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4",
}

missing_models = []
for tag, mid in MODELS.items():
    try:
        info = model_info(mid)
        n = len([s for s in info.siblings
                 if s.rfilename.endswith((".safetensors", ".bin"))])
        print("  %-12s %-45s 权重文件 %d 个  OK" % (tag, mid, n))
    except Exception as e:
        missing_models.append(tag)
        print("  %-12s %-45s 取不到: %s" % (tag, mid, type(e).__name__))

print()
if missing_models:
    print("这些仓库取不到:", missing_models)
    print("把这一行发我，我换等价的量化仓库，别硬跑。")
else:
    print("三个模型都在，可以往下跑。")


## 3. 写出压测脚本（与前几轮同一份，客户端一行不改）

In [ ]:
import io
src = '# -*- coding: utf-8 -*-\n"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。\n\n指标定义（与 JD 里那套一致）：\n  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感\n  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔\n  吞吐   总输出 token 数 / 墙钟时间\n\n用法（先 bash serve.sh 起服务）：\n  python bench_serving.py                 # 并发扫描\n  python bench_serving.py --prefix-test   # 前缀复用对照\n"""\nimport argparse, asyncio, json, statistics as st, time\nimport aiohttp\n\nURL = "http://127.0.0.1:8000/v1/chat/completions"\nMODEL = "Qwen/Qwen2.5-0.5B-Instruct"\n\n# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次\nSHARED_PREFIX = (\n    "You are a meticulous technical assistant. Answer concisely and precisely. "\n    "Always reason step by step before answering. " * 20\n)\n\n\nasync def one_request(sess, prompt, max_tokens, use_prefix):\n    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \\\n           [{"role": "user", "content": prompt}]\n    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,\n            "temperature": 0.0, "stream": True}\n    t0 = time.perf_counter()\n    ttft, n_tok, last = None, 0, t0\n    async with sess.post(URL, json=body) as resp:\n        async for raw in resp.content:\n            line = raw.decode("utf-8").strip()\n            if not line.startswith("data: ") or line == "data: [DONE]":\n                continue\n            delta = json.loads(line[6:])["choices"][0].get("delta", {})\n            if delta.get("content"):\n                now = time.perf_counter()\n                if ttft is None:\n                    ttft = now - t0\n                n_tok += 1\n                last = now\n    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)\n\n\nasync def run_batch(n_conc, n_req, max_tokens, use_prefix):\n    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]\n    sem = asyncio.Semaphore(n_conc)\n\n    async def guarded(sess, p):\n        async with sem:\n            return await one_request(sess, p, max_tokens, use_prefix)\n\n    timeout = aiohttp.ClientTimeout(total=600)\n    async with aiohttp.ClientSession(timeout=timeout) as sess:\n        await one_request(sess, "warmup", 4, use_prefix)          # 预热\n        t0 = time.perf_counter()\n        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))\n        wall = time.perf_counter() - t0\n\n    tot_tok = sum(r["n_tok"] for r in rs)\n    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]\n    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,\n                ttft_p50=st.median(r["ttft"] for r in rs),\n                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],\n                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)\n\n\nasync def sweep(args):\n    print(f"{\'并发\':>5}{\'请求\':>6}{\'墙钟s\':>9}{\'吞吐 tok/s\':>13}{\'RPS\':>8}"\n          f"{\'TTFT p50\':>11}{\'TTFT p99\':>11}{\'TPOT p50\':>11}")\n    print("-" * 74)\n    out = []\n    for c in [1, 2, 4, 8, 16, 32]:\n        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)\n        print(f"{r[\'conc\']:>5}{max(c*4,16):>6}{r[\'wall\']:>9.2f}{r[\'tput\']:>13.1f}"\n              f"{r[\'rps\']:>8.2f}{r[\'ttft_p50\']*1e3:>10.1f}ms{r[\'ttft_p99\']*1e3:>10.1f}ms"\n              f"{r[\'tpot_p50\']*1e3:>10.2f}ms")\n        out.append(r)\n    json.dump(out, open("sweep_results.json", "w"), indent=1)\n    base = out[0]["tput"]\n    print(f"\\ncontinuous batching 收益：并发 1 → 32，吞吐 "\n          f"{base:.1f} → {out[-1][\'tput\']:.1f} tok/s（{out[-1][\'tput\']/base:.1f}×），"\n          f"TTFT p50 {out[0][\'ttft_p50\']*1e3:.0f} → {out[-1][\'ttft_p50\']*1e3:.0f} ms")\n    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")\n\n\nasync def prefix_test(args):\n    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")\n    print(f"{\'场景\':<26}{\'吞吐 tok/s\':>13}{\'TTFT p50\':>12}")\n    print("-" * 51)\n    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:\n        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)\n        print(f"{label:<26}{r[\'tput\']:>13.1f}{r[\'ttft_p50\']*1e3:>11.1f}ms")\n    print("\\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")\n    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--max-tokens", type=int, default=128)\n    ap.add_argument("--prefix-test", action="store_true")\n    a = ap.parse_args()\n    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))\n'
# 模型名改成变量，由启动器写入 MODEL_NAME.txt
src = src.replace('MODEL = "Qwen/Qwen2.5-0.5B-Instruct"',
                  'import os; MODEL = open("MODEL_NAME.txt").read().strip()')
io.open("bench_serving.py", "w", encoding="utf-8").write(src)
print("写出 bench_serving.py", len(src), "字符（模型名改为从 MODEL_NAME.txt 读）")


## 3b. 探测 `--quantization` 的合法取值

不凭记忆写参数——SGLang 那轮就是靠这一步，第一时间发现问题不在参数名而在依赖层。
这一格把当前 vLLM 版本 `--quantization` 的取值列表打出来，确认 `awq` / `gptq` 真的在里面。

**若某个取值不存在，不要往下跑**，把输出发我换等价名称（如 `awq_marlin` / `gptq_marlin`）。


In [ ]:
import subprocess, sys, re

r = subprocess.run([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--help"],
                   capture_output=True, text=True)
h = r.stdout + r.stderr
print("help returncode:", r.returncode, "| 长度:", len(h))

m = re.search(chr(45)+chr(45)+"quantization" + "[^" + chr(10) + "]*?" + chr(92)+"{([^}]*)" + chr(92)+"}", h)
vals = m.group(1).split(",") if m else []
print("--quantization 取值:", vals or "（没抓到取值列表）")
print()
for want in ["awq", "gptq"]:
    print("  %-8s %s" % (want, "存在 OK" if want in vals else "不在列表 !!"))
print()
print("相关取值（含 marlin 等变体）:", [v for v in vals if "awq" in v or "gptq" in v])
print()
print("结论:", "可以往下跑。" if all(w in vals for w in ["awq","gptq"]) else "有取值对不上，先停，把上面发我。")


## 4. 启动器 + 一轮完整测量

每个变体：起服务 → 抓显存与 KV cache 行 → 并发扫描 → 关服务。
关掉再起下一个，**不让两份权重同时占显存**——
上一轮我就是让三份实现同时驻留，测出 137 ms 的假象，隔离后实际 17.9 ms。


In [ ]:
import subprocess, sys, time, requests, io, re, json, os

RESULTS = {}

def serve(model_id, quant, tag, wait=420):
    subprocess.run(["pkill", "-f", "vllm.entrypoints"], check=False)
    subprocess.run(["pkill", "-f", "vllm serve"], check=False)
    time.sleep(10)
    io.open("MODEL_NAME.txt", "w").write(model_id)

    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model_id,
           "--host", "127.0.0.1", "--port", "8000",
           "--max-model-len", "2048",
           "--gpu-memory-utilization", "0.85",
           "--no-enable-log-requests"]   # vLLM 0.28 已把 --disable-log-requests 改名，用旧名 argparse 退出码 2
    if quant:
        cmd += ["--quantization", quant]

    print("启动:", model_id, "| quantization =", quant or "（无，fp16）")
    log = open("/content/vllm_%s.log" % tag, "w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

    for i in range(wait // 2):
        if p.poll() is not None:
            print("  进程退出（码 %s），日志尾部：" % p.returncode)
            print(open("/content/vllm_%s.log" % tag).read()[-3500:])
            return None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("  就绪，用时 %ds" % (i * 2))
                txt = open("/content/vllm_%s.log" % tag).read()
                info = {}
                for pat, key in [(r"GPU KV cache size: ([\d,]+)", "kv_tokens"),
                                 (r"model weights take ([\d.]+)\s*GiB", "weights_gib"),
                                 (r"(Maximum concurrency[^\n]*)", "concurrency")]:
                    m = re.search(pat, txt)
                    if m:
                        info[key] = m.group(1)
                        print("   ", key, "=", m.group(1))
                RESULTS.setdefault(tag, {})["serve_info"] = info
                return p
        except requests.RequestException:
            pass          # 只吞连接失败；其它异常照抛，别再无声吞掉真 bug
        time.sleep(2)

    print("  %ds 没起来，日志尾部：" % wait)
    print(open("/content/vllm_%s.log" % tag).read()[-3500:])
    return None


def bench(tag):
    r = subprocess.run([sys.executable, "-u", "bench_serving.py"],
                       capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        return
    if os.path.exists("sweep_results.json"):
        RESULTS.setdefault(tag, {})["sweep"] = json.load(open("sweep_results.json"))
        os.rename("sweep_results.json", "sweep_%s.json" % tag)


### 4a. fp16 基线

In [ ]:
p = serve(MODELS["fp16"], None, "fp16")
if p: bench("fp16")

### 4b. AWQ

In [ ]:
p = serve(MODELS["AWQ"], "awq", "awq")
if p: bench("awq")

### 4c. GPTQ-Int4

In [ ]:
p = serve(MODELS["GPTQ-Int4"], "gptq", "gptq")
if p: bench("gptq")

## 5. 精度：wikitext 困惑度

PPL 是量化论文里的标准精度指标。三个变体用**同一段文本、同一个窗口长度**算，
差值才可比。跑在子进程里，算完就释放显存。

注意：PPL 低不等于任务能力强，它只说明「语言建模分布偏移了多少」。
这一点要写进结论，别把 PPL 当成任务准确率。


In [ ]:
ppl_script = r'''
import sys, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = sys.argv[1]
tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda",
                                             torch_dtype="auto", trust_remote_code=True)
model.eval()

from datasets import load_dataset
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n\n".join(ds["text"])[:200000]
enc = tok(text, return_tensors="pt").input_ids

WIN, STRIDE = 1024, 1024
nlls, n = [], 0
with torch.no_grad():
    for i in range(0, min(enc.size(1), WIN * 40) - WIN, STRIDE):
        ids = enc[:, i:i+WIN].cuda()
        out = model(ids, labels=ids)
        nlls.append(out.loss.float() * (WIN - 1))
        n += WIN - 1
ppl = torch.exp(torch.stack(nlls).sum() / n).item()
print(json.dumps({"model": model_id, "ppl": round(ppl, 4), "windows": len(nlls)}))
'''
io.open("ppl.py", "w", encoding="utf-8").write(ppl_script)

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
subprocess.run(["pkill", "-f", "vllm"], check=False)
import time; time.sleep(10)

PPL = {}
for tag, mid in MODELS.items():
    print("=== PPL:", tag, mid, "===")
    r = subprocess.run([sys.executable, "ppl.py", mid], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("{")]
    if line:
        d = json.loads(line[-1]); PPL[tag] = d["ppl"]
        print("   PPL =", d["ppl"], "（%d 个窗口）" % d["windows"])
    else:
        print("   失败：", (r.stderr or r.stdout)[-1500:])
print()
print("PPL 汇总:", PPL)


## 6. 汇总

In [ ]:
import json

print("=" * 78)
hdr = "%-12s %12s %12s %12s %10s %10s" % ("变体", "吞吐@1", "吞吐@8", "吞吐@32", "TTFT@8", "PPL")
print(hdr)
print("-" * 78)

def pick(sweep, conc):
    for r in sweep or []:
        if r["conc"] == conc:
            return r
    return None

base = None
for tag in ["fp16", "awq", "gptq"]:
    sw = RESULTS.get(tag, {}).get("sweep")
    r1, r8, r32 = pick(sw, 1), pick(sw, 8), pick(sw, 32)
    key = {"fp16": "fp16", "awq": "AWQ", "gptq": "GPTQ-Int4"}[tag]
    print("%-12s %12s %12s %12s %10s %10s" % (
        key,
        "%.1f" % r1["tput"] if r1 else "-",
        "%.1f" % r8["tput"] if r8 else "-",
        "%.1f" % r32["tput"] if r32 else "-",
        "%.0fms" % (r8["ttft_p50"] * 1e3) if r8 else "-",
        PPL.get(key, "-"),
    ))
    if tag == "fp16" and r8:
        base = r8["tput"]

print("-" * 78)
if base:
    for tag, key in [("awq", "AWQ"), ("gptq", "GPTQ-Int4")]:
        r8 = pick(RESULTS.get(tag, {}).get("sweep"), 8)
        if r8:
            print("%s 相对 fp16（并发 8）: %.3f 倍" % (key, r8["tput"] / base))

print()
print("显存与 KV cache:")
for tag in ["fp16", "awq", "gptq"]:
    print("  %-8s" % tag, RESULTS.get(tag, {}).get("serve_info", {}))

json.dump({"results": RESULTS, "ppl": PPL}, open("quant_matrix.json", "w"),
          ensure_ascii=False, indent=1)
print()
print("已存 quant_matrix.json")


## 7. 怎么写结论

### 先回看第 0 节那个预测

预测是「T4 无 Marlin，GPTQ-Int4 吞吐可能不如 fp16」。
- **若数据支持** —— 这是一条有价值的发现：量化省显存但在该架构上不省时间，
  和本仓「无 Tensor Core 卡上 fp32 快过 fp16」是同一类结论，可以合并成一句话讲。
- **若数据推翻** —— 照写推翻，说明 vLLM 在 sm_75 上的 GPTQ 路径比预期好，
  预测错了就是错了。**不要事后改预测去迎合数据。**

### 必须写进边界的几条

- PPL 只反映语言建模分布的偏移，**不是任务准确率**。要谈任务能力得另跑
  LongBench 之类的基准，本轮没跑。
- 用的是社区已量化好的权重，**量化过程本身不是我做的**——
  简历只能写「做了 PTQ 变体的性能与精度对照」，不能写「实现了量化算法」。
- QAT 完全没做。
- 单卡 T4、1.5B 模型、2048 上下文，结论不能外推到大模型或多卡。
